In [9]:
import json

internvl_data_path = "dataset.jsonl"

with open(internvl_data_path, "r") as f:
    data = f.readlines()
    data = [json.loads(line) for line in data]

print(data[100]["conversations"])
print(data[100]["answer"])
print(data[100]["image_urls"])
print(data[1000].keys())
print(len(data))

[{'role': 'system', 'content': '你是书生·万象，英文名是InternVL，是由上海人工智能实验室、清华大学及多家合作单位联合开发的多模态大语言模型。'}, {'role': 'user', 'content': '<image>\nYou should first thinks about the reasoning process in the mind and then provides the user with the answer. Your answer must be in latex format and wrapped in $...$.The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> Since $1+1=2$, so the answer is $2$. </think><answer> $2$ </answer>, which means your output should start with <think> and end with </answer>.\nQuestion:\nHow many shapes are there?\nA. 5\nB. 3\nC. 2\nD. 4\nE. 1'}]
$A$
['MMPR/images/iconqa/iconqa_data/iconqa/train/choose_txt/85488/image.png']
dict_keys(['id', 'conversations', 'answer', 'image_urls'])
54931


In [10]:
"""
{
"question": "How many shapes are blue?",
"answer": "$4$",
"message": "[{"role": "system", "content": "Solve the question. The user asks a question, and you solves it. You should first provide a detailed description of the image if an image is present. If no image is provided, rephrase and elaborate on the question in a different way, which in the format ..., followed by the reasoning process in the mind within ..., and then provide the answer within .... Your answer must be in latex format and wrapped in $...$. The reasoning process and answer are enclosed within and tags, respectively, i.e., ... Since 
 . Your output should start with and end with ."}, {"role": "user", "content": [{"type": "text", "text": "How many shapes are blue?"}, {"type": "image", "image": "/mnt/train/fill_in_blank/58681/image.png"}]}]"
}
"""
def transfer2qwen(item):
    if len(item["image_urls"]) != 1:
        return None
    question = item["conversations"][1]["content"].split("\nQuestion:\n")[1]
    answer = item["answer"]
    system_prompt = "Solve the question. The user asks a question, and you solves it. You should first thinks about the reasoning process in the mind and then provides the user with the answer. Your answer must be in latex format and wrapped in $...$. The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> Since $1+1=2$, so the answer is $2$. </think><answer> $2$ </answer>, which means your output should start with <think> and end with </answer>."
    image_path = item["image_urls"][0]
    image_path = f"/workspace/Jiawei/Datasets/MM-Eureka-Dataset/{image_path}"

    message = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {"type": "image", "image": image_path},
            ],
        },
    ]

    new_item = {
        "question": question,
        "answer": answer,
        "message": json.dumps(message)
    }

    return new_item

new_data = []
for item in data:
    new_item = transfer2qwen(item)
    if new_item is not None:
        new_data.append(new_item)

with open("dataset_qwen.jsonl", "w") as f:
    for item in new_data:
        f.write(json.dumps(item) + "\n")

In [11]:
new_data[0]

{'question': 'As shown in the figure, AD is the median of △ABC, and E is the midpoint of AD. The area of △ABE is 2, then the area of △ABC is ( ).',
 'answer': '$8$',
 'message': '[{"role": "system", "content": "Solve the question. The user asks a question, and you solves it. You should first thinks about the reasoning process in the mind and then provides the user with the answer. Your answer must be in latex format and wrapped in $...$. The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> Since $1+1=2$, so the answer is $2$. </think><answer> $2$ </answer>, which means your output should start with <think> and end with </answer>."}, {"role": "user", "content": [{"type": "text", "text": "As shown in the figure, AD is the median of \\u25b3ABC, and E is the midpoint of AD. The area of \\u25b3ABE is 2, then the area of \\u25b3ABC is ( )."}, {"type": "image", "image": "/workspace/Jiawei/Datasets/MM-Eureka-Dataset/MMPR